In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import numpy as np
import sklearn

## Preprocessing / Extraction

- get all require, assert and revert statements
- strip them of their surrounding strucutre and only keep `predicate, (errormessage)`
- for those where the error message does not exist, try to get it from the error reason
- delete assert, require, revert keywords

In [29]:
def find_balanced_expr(s, start, open_char='(', close_char=')'):
        if s[start] != open_char:
            return None
        depth = 0
        for i in range(start, len(s)):
            if s[i] == open_char:
                depth += 1
            elif s[i] == close_char:
                depth -= 1
                if depth == 0:
                    return s[start+1:i]
        return None

def invert_predicate(predicate):
    """Intelligently invert a basic logical predicate."""
    predicate = predicate.strip()

    # Single condition logic
    comparison_operators = {
        '==': '!=',
        '!=': '==',
        '<=': '>',
        '>=': '<',
        '<': '>=',
        '>': '<='
    }

    # Try to invert simple comparison expressions
    for op in sorted(comparison_operators, key=len, reverse=True):  # Match >= before >
        if op in predicate:
            parts = predicate.split(op)
            if len(parts) == 2:
                lhs, rhs = parts[0].strip(), parts[1].strip()
                inverted_op = comparison_operators[op]
                return f"{lhs} {inverted_op} {rhs}"

    # If we have multiple conditions (&& or ||), or anything not matched above
    if '&&' in predicate or '||' in predicate:
        return f"!({predicate})"  # Fallback for compound conditions

    # For unary negation
    if predicate.startswith('!'):
        return predicate[1:].strip('() ')  # !flag → flag

    return f"!({predicate})"  # Fallback catch-all


In [30]:
def extract_predicate_and_error_after_revert(line):
    # Step 1: Find distinct 'if'
    if line.count('if') != 1:
        return None, None

    if_index = line.find('if')
    if if_index == -1:
        return None, None

    # Step 2: Get start of condition
    paren_start = line.find('(', if_index)
    if paren_start == -1:
        return None, None

    # Step 3: Get the balanced predicate
    predicate = find_balanced_expr(line, paren_start)
    if predicate is None:
        return None, None

    # Step 4: Find closing paren index
    depth = 0
    for i in range(paren_start, len(line)):
        if line[i] == '(':
            depth += 1
        elif line[i] == ')':
            depth -= 1
            if depth == 0:
                after_if_pos = i + 1
                break
    else:
        return None, None

    # Step 5: Find `revert` after `if`
    revert_index = line.find('revert', after_if_pos)
    if revert_index == -1:
        return None, None

    # Step 6: Check if 'else' is between `if` block and `revert`
    segment = line[after_if_pos:revert_index]
    if 'else' in segment:
        effective_predicate = invert_predicate(predicate.strip())
    else:
        effective_predicate = predicate.strip()

    # Step 7: Extract error message from revert(...)
    revert_paren_start = line.find('(', revert_index)
    if revert_paren_start == -1:
        return effective_predicate, None

    error_message = find_balanced_expr(line, revert_paren_start)
    if error_message:
        error_message = error_message.strip().strip('"\'')  # remove quotes
    else:
        error_message = None


    return effective_predicate, error_message


In [31]:
import re

def extract_revert_parts(statement):
    statement = statement.strip()

    # --- 1. Match inline: if (...) revert Error(...);
    inline = re.match(
        r'^if\s*\((.*?)\)\s*revert\s+([a-zA-Z0-9_.]+)\s*\((.*?)\)?\s*;?',
        statement, re.IGNORECASE | re.DOTALL
    )
    if inline:
        predicate = inline.group(1).strip()
        error = inline.group(2).strip()
        args = inline.group(3).strip()
        return predicate,  f"{error}({args})"

    # --- 2. Match block: if (...) { revert Error(...) }
    block = re.match(
        r'^if\s*\((.*?)\)\s*\{\s*revert\s+([a-zA-Z0-9_.]+)\s*\((.*?)\)?\s*;?',
        statement, re.IGNORECASE | re.DOTALL
    )
    if block:
        predicate = block.group(1).strip()
        error = block.group(2).strip()
        args = block.group(3).strip()
        return predicate,  f"{error}({args})"

    # --- 3. Match simple: if (...) { revert(); }
    simple_block = re.match(
        r'^if\s*\((.*?)\)\s*\{\s*revert\s*\(\s*\)\s*;?',
        statement, re.IGNORECASE | re.DOTALL
    )
    if simple_block:
        predicate = simple_block.group(1).strip()
        return predicate, None

    # --- 4. Match inline string revert: if (...) revert("string message");
    inline_str = re.match(
        r'^if\s*\((.*?)\)\s*revert\s*\(\s*"(.*?)"\s*\)\s*;?',
        statement, re.IGNORECASE | re.DOTALL
    )
    if inline_str:
        predicate = inline_str.group(1).strip()
        message = inline_str.group(2).strip()
        return predicate, message

    # --- 5. Match block string revert: if (...) { revert("string message"); }
    block_str = re.match(
        r'^if\s*\((.*?)\)\s*\{\s*revert\s*\(\s*"(.*?)"\s*\)\s*;?\s*\}',
        statement, re.IGNORECASE | re.DOTALL
    )
    if block_str:
        predicate = block_str.group(1).strip()
        message = block_str.group(2).strip()
        return predicate, message
    
    predicate, error = extract_predicate_and_error_after_revert(statement)
    if predicate is not None:
        return predicate, error
    
    return None, None


In [32]:
import pandas as pd
import re

def preprocessing1(parquet_file_path, save_dir = "../datasets/master_thesis_20000", with_message=True):
    df = pd.read_parquet(parquet_file_path)
    df = df[df['failure_invariant'].notna()].copy()
    df['failure_invariant_str'] = df['failure_invariant'].astype(str).str.lower().str.strip()
    assert_require_mask = df['failure_invariant_str'].str.contains(r'\b(assert|require)\b', regex=True, na=False)
    revert_with_if_mask = df['failure_invariant_str'].str.contains(r'\brevert\b', regex=True, na=False) & \
                        df['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    throw_with_if_mask = df['failure_invariant_str'].str.contains(r'\bthrow\b', regex=True, na=False) & \
                        df['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    final_mask = assert_require_mask | revert_with_if_mask | throw_with_if_mask
    df = df[final_mask].copy()
    df = df.drop_duplicates(subset='failure_invariant_str').reset_index(drop=True)
    unique_invariants = df['failure_invariant_str'].drop_duplicates().reset_index(drop=True)
    print(len(df))
    #print(df['failure_invariant'].to_string(index=False))


    def split_condition_and_error(expr):
        """Split a require/assert argument into (predicate, error)"""
        in_quote = False
        escape = False
        depth = 0
        for i, ch in enumerate(expr):
            if ch in ['"', "'"] and not escape:
                in_quote = not in_quote
            elif ch == ',' and not in_quote and depth == 0:
                return expr[:i].strip(), expr[i+1:].strip()
            elif ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
            escape = (ch == '\\')
        return expr.strip(), None


    def extract_parts(statement):
        statement = statement.strip()
        pattern = r'^(\}\s*)else(\s+if.*)'
        match = re.match(pattern, statement)
        if match:
            statement = match.group(1) + match.group(2)  # drop 'else', keep '} if (...)'


        # Handle require(...) and assert(...) with balanced parentheses or inline blocks / if statements
        m = re.match(r'^(require|assert)\s*\(', statement)
        if m:
            open_idx = statement.find('(', m.end() - 1)
            if open_idx != -1:
                inner = find_balanced_expr(statement, open_idx)
                if inner:
                    cond, error = split_condition_and_error(inner)
                    if error and with_message:
                        error = error.strip().strip('\'"')
                        return cond, error
                    return cond, None

        # Handle require inside inline code blocks or after if condition without braces
        m = re.search(r'if\s*\((.+?)\)\s*\{[^}]*require\s*\((.+?)\);\s*\}', statement)
        if m:
            req_cond, req_error = split_condition_and_error(m.group(2).strip())
            if req_error and with_message:
                req_error = req_error.strip().strip('\'"')
                return req_cond, req_error
            return req_cond, None

        m = re.search(r'\{\s*require\s*\((.+?)\);', statement)
        if m:
            cond, error = split_condition_and_error(m.group(1).strip())
            if error and with_message:
                error = error.strip().strip('\'"')
                return cond, error
            return cond, None

        m = re.search(r'if\s*\((.+?)\)\s*require\s*\((.+?)\);', statement)
        if m:
            req_cond, req_error = split_condition_and_error(m.group(2).strip())
            if req_error and with_message:
                req_error = req_error.strip().strip('\'"')
                return req_cond, req_error
            return req_cond , None

        m = re.match(r'require\s*\((.+?)\);', statement)
        if m:
            cond, error = split_condition_and_error(m.group(1))
            if error and with_message:
                error = error.strip().strip('\'"')
                return cond, error
            return cond, None

        # Existing assert matches
        m = re.match(r'^assert\s+(.+?)\s*,\s*["\'](.+?)["\'](?:\s*#.*)?$', statement)
        if m:
            predicate = m.group(1).strip()
            error = m.group(2).strip()
            if error and with_message:
                return predicate, error
            return predicate, None

        m = re.match(r'^assert\s+(.+)$', statement)
        if m:
            predicate = m.group(1).strip()
            return predicate, None

        m = re.match(r'^if\s*\((.*?)\)\s*throw\s*;', statement.strip())
        if m:
            predicate = m.group(1).strip()
            return predicate, None

        # Try revert if/else revert extraction
        revert_res, error = extract_revert_parts(statement)
        if revert_res and error:
            return revert_res, error
        if revert_res:
            return revert_res, None

        # No match
        print("UNMATCHED:", statement)
        return None, None

    df[['predicate', 'message']] = pd.DataFrame(df['failure_invariant_str'].apply(extract_parts).tolist(), index=df.index)
    df = df[df['predicate'].notnull()]
    if 'gas_used' in df.columns and 'gas_price' in df.columns:
        df['gas_loss_eth'] = (df['gas_used'] * df['gas_price']) / 1e18

    df[['message']] = None
    if 'failure_reason' in df.columns:
        mask_reason = df['failure_reason'].notna() & ~df['failure_reason'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_reason, 'message'] = df.loc[mask_reason, 'failure_reason']

    # If 'failure_reason' doesn't exist or you want to use 'failure_message' as a fallback
    if 'failure_message' in df.columns:
        mask_message = df['failure_message'].notna() & ~df['failure_message'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_message, 'message'] = df.loc[mask_message, 'failure_message']


    result = df[['predicate', 'message']].drop_duplicates(subset=['predicate']).reset_index(drop=True)

    result.to_csv(f"{save_dir}_extracted.csv", index=False, header=True)

    

    return result


In [33]:
# Create a string for the error message
def df_to_string(df, with_message=True):
    lines = []
    for _, row in df.iterrows():
        predicate = row['predicate']
        message = row['message']
        if with_message and pd.notnull(message):
            line = f"{predicate}, '{message}'"
        else:
            line = f"{predicate},"
        lines.append(line)
    return lines

## Embedding
- embed the transformer models

In [34]:
from transformers import RobertaTokenizer, RobertaModel
import torch
import numpy as np
from sklearn.preprocessing import Normalizer, normalize
from sklearn.decomposition import PCA
from umap import UMAP
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def embedding_model(texts, name):
    tokenizer = RobertaTokenizer.from_pretrained(name)
    model = RobertaModel.from_pretrained(name)
    model.eval()
    embeddings = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
        embeddings.append(cls_embedding)
    the_embeddings = np.array(embeddings)
    return the_embeddings

def normalize_and_reduce(data, pca_components=0.95):
    """
    Normalize data with L2 norm and reduce dimensionality using PCA.
    """
    #reduced_data = UMAP(n_neighbors=15, n_components=5, metric="cosine").fit_transform(data)
    reduced_data = UMAP(n_components=20, random_state=42, metric = "cosine").fit_transform(data)
    #embeddings_normalized = normalize(data, norm='l2') 
    #reduced_data = StandardScaler().fit_transform(embeddings_normalized)
    """
    pipeline = make_pipeline(
        Normalizer(norm='l2'),  # L2 normalization
        PCA(n_components=10)
    )

    # Apply to data
    reduced_data = pipeline.fit_transform(data)
    """
    return reduced_data

## Clustering and Metrics

In [35]:
def print_clusters(labels, input_values, algorithm,model, output_file="clusters_output.txt", skip_noise=True):
    from collections import defaultdict

    cluster_dict = defaultdict(list)
    for label, value in zip(labels, input_values):
        cluster_dict[label].append(value)

    with open(f"{output_file}/{model}_{algorithm}.txt", "w", encoding="utf-8") as f:
        f.write(f"Clusters generated for {algorithm} and {model}\n")
        f.write("=" * 60 + "\n")

        for cluster_id in sorted(cluster_dict.keys()):
            if skip_noise and cluster_id == -1:
                continue
            f.write(f"\n--- Cluster {cluster_id} ({len(cluster_dict[cluster_id])} items) ---\n")
            for item in cluster_dict[cluster_id]:
                f.write(f"- {item}\n")

        if -1 in cluster_dict and not skip_noise:
            f.write(f"\n--- Noise / Outliers (label = -1) ({len(cluster_dict[-1])} items) ---\n")
            for item in cluster_dict[-1]:
                f.write(f"- {item}\n")

    print(f"\nClusters saved to {output_file}")


In [36]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer

import matplotlib.cm as cm
def vislualize_cluster(data, labels, cluster_length, algorithm, model):
    colors = cm.get_cmap('viridis', cluster_length)
    for i in range(cluster_length):
        plt.scatter(data[labels == i, 0], data[labels == i, 1], 
                    color=colors(i), label=f'Cluster {i}', s=40)
    plt.scatter(data[labels == -1, 0], data[labels == -1, 1], 
                color='gray', label='Noise / Outliers', alpha=0.5, s=25)

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f'{algorithm} Clustering: {cluster_length} Clusters for {model}')
    plt.tight_layout()
    plt.figure(figsize=(10, 6))

    plt.show()


In [37]:
POINTS_THRESHOLD = 0.5
MIN_CLUSTER = 8
MAX_CLUSTER = 100

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import cosine_distances
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm
import hdbscan
from umap import UMAP
import os
import numpy as np
from s_dbw import S_Dbw

def cluster_evaluate(labels, data, model_name, metric="euclidean"):
    labels = np.array(labels)
    valid_mask = labels != -1
    valid_data = data[valid_mask]
    valid_labels = labels[valid_mask]

    pct_clustered = len(valid_labels) / len(labels)
    num_clusters = len(set(valid_labels)) if len(set(valid_labels)) > 1 else 0
    sil_score = -1
    sdbw_score = 100

    if pct_clustered > POINTS_THRESHOLD and num_clusters > MIN_CLUSTER and num_clusters < MAX_CLUSTER:
        sil_score = silhouette_score(valid_data, valid_labels, metric=metric)
        sdbw_score = S_Dbw(valid_data, valid_labels, metric = "cosine", alg_noise='bind', centr='mean')

        sil_score = round(sil_score, 2)
    result = {
        "embedding_model": model_name,
        "metric": metric,
        "silhouette": sil_score,
        "sdbw": sdbw_score,
        "num_clusters": len(set(valid_labels)) if len(set(valid_labels)) > 1 else 0,
        "pct_clustered": round( 100 * len(valid_labels) / len(labels), 2),
        "labels": labels,
        }

    return result

def cluster_search_kmeans(data, model_name, metric="cosine", k_range=range(MIN_CLUSTER, MAX_CLUSTER)):
    best_result, best_k = None, None
    for k in tqdm(k_range, desc=f"KMeans-{metric}"):
        clusterer = KMeans(n_clusters=k, random_state=42, max_iter=100, init="k-means++", tol=1e-6)
        labels = clusterer.fit_predict(data)
        scores = cluster_evaluate(labels, data, model_name, metric=metric)
        if not best_result or scores["sdbw"] < best_result["sdbw"]:
            best_result, best_k = scores, k
    if best_result:
        best_result.update({"algorithm": "KMeans", "params": {"n_clusters": best_k}})
    return best_result

def cluster_search_dbscan(data, model_name, metric="cosine", eps_values=np.linspace(0.1, 5, 20), min_samples_range=range(10, 20)):
    best_result, best_params = None, None
    for eps in tqdm(eps_values, desc=f"DBSCAN-{metric}"):
        for min_samples in min_samples_range:
            try:
                clusterer = DBSCAN(eps=eps, min_samples=min_samples)
                labels = clusterer.fit_predict(data)
                eval_data = data
                scores = cluster_evaluate(labels, eval_data, model_name, metric=metric)
                if not best_result or scores["sdbw"] < best_result["sdbw"]:
                    best_result, best_params = scores, {"eps": eps, "min_samples": min_samples}
            except Exception:
                continue
    if best_result:
        best_result.update({"algorithm": "DBSCAN", "params": best_params})
    return best_result

def cluster_search_hdbscan(data, model_name, metric="cosine", min_cluster_range=range(10, 20)):
    best_result, best_param = None, None
    for min_cluster_size in tqdm(min_cluster_range, desc=f"HDBSCAN-{metric}"):
        try:
            if metric == "cosine":
                cosine_dist = cosine_distances(data).astype(np.float64)
                clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric='precomputed')
                labels = clusterer.fit_predict(cosine_dist)
                eval_data = cosine_dist
            else:
                clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric=metric, random_state=42)
                labels = clusterer.fit_predict(data)
                eval_data = data
            scores = cluster_evaluate(labels, eval_data, model_name, metric=metric)
            if not best_result or scores["sdbw"] < best_result["sdbw"]:
                best_result, best_param = scores, {"min_cluster_size": min_cluster_size}
        except Exception as e:
            print(f"Exception: {e}")
            continue

    if best_result:
        best_result.update({"algorithm": "HDBSCAN", "params": best_param})
    return best_result

# -------------------------------
# Main Runner Across Metrics
# -------------------------------

def cluster_run_all_with_metrics(data, model_name, original_data, output_file="clustering_results_all_metrics.csv", with_msg = False, metrics=["cosine", "manhattan", "levensthein", "euclidean"]):
    all_results = []
    folder_msg = "experiments/without"
    if with_msg:
        folder_msg = "experiments/with"

    data = normalize_and_reduce(data)

    for metric in metrics:
        print(f"\n=== Running clustering for metric: {metric} ===")
        try:
            kmeans_result = cluster_search_kmeans(data, model_name, metric=metric)
            if kmeans_result: 
                if kmeans_result["silhouette"] > 0:
                    all_results.append(kmeans_result)
                    print_clusters(kmeans_result["labels"], original_data, model_name, "kmeans", output_file = folder_msg, skip_noise=False)
                    vislualize_cluster(data, kmeans_result["labels"], kmeans_result["num_clusters"], "K-Means", model_name)

            dbscan_result = cluster_search_dbscan(data, model_name, metric=metric)
            if dbscan_result: 
                if dbscan_result["silhouette"] > 0:
                    all_results.append(dbscan_result)
                    print_clusters(dbscan_result["labels"], original_data, model_name, "dbscan", output_file = folder_msg, skip_noise=True)
                    vislualize_cluster(data, dbscan_result["labels"], dbscan_result["num_clusters"], "DBSCAN", model_name)

            hdbscan_result = cluster_search_hdbscan(data, model_name, metric=metric)
            if hdbscan_result: 
                if hdbscan_result["silhouette"] > 0:
                    all_results.append(hdbscan_result)
                    print_clusters(hdbscan_result["labels"], original_data, model_name, "hdbscan", output_file = folder_msg, skip_noise=True)
                    vislualize_cluster(data, hdbscan_result["labels"], hdbscan_result["num_clusters"], "HDBSCAN", model_name)

        except Exception as e:
            print(f"Error processing metric {metric}: {e}")
            continue

    df = pd.DataFrame(all_results)
    if 'labels' in df.columns:
        df = df.drop(columns=['labels'])

    write_mode = 'a' if os.path.exists(output_file) else 'w'
    df.to_csv(output_file, mode=write_mode, index=False)
    selected_columns = [
        "embedding_model",
        "metric",
        "algorithm",
        "params",
        "combined_scores",
        "silhouette",
        "davies_bouldin",
        "calinski_harabasz",
        "num_clusters",
        "pct_clustered"
    ]
    selected_columns = [col for col in selected_columns if col in df.columns]

    print("\nFinal Results (Selected Columns):")
    print(df[selected_columns].to_string(index=False))
    print(f"\nSaved to {output_file}")

    return df


In [13]:
def string_length_stats(df, with_message=True):
    strings = df_to_string(df, with_message=with_message)
    lengths = [len(s) for s in strings]

    if not lengths:
        return {"average": 0, "min": 0, "max": 0}

    average_length = sum(lengths) / len(lengths)
    min_length = min(lengths)
    max_length = max(lengths)

    return {
        "average": average_length,
        "min": min_length,
        "max": max_length
    }


## Run the clustering pipeline
- run the clustering for the unique invariants for with and without error messages

## Extract the invariants

In [20]:
# to run the extraction for the fine tuned dataset:
# file_path = "../datasets/finetuning_dataset.parquet"x
#df = pd.read_parquet(file_path)

# df = preprocessing1(file_path, out_)


In [27]:
file_path = "../datasets/master_thesis_20000.parquet"
df = pd.read_parquet(file_path)

df = preprocessing1(file_path)
print(len(df))
count_with_message = df['message'].notna().sum()

count_without_message = df['message'].isna().sum()
print(count_with_message)
print(count_without_message)
with_message = df_to_string(df)
without_message = df_to_string(df, with_message=False)
print(string_length_stats(df))
print(string_length_stats(df, with_message = False))

C:\Users\Melis\AppData\Local\Temp\ipykernel_39264\1454680835.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  assert_require_mask = df['failure_invariant_str'].str.contains(r'\b(assert|require)\b', regex=True, na=False)


838
UNMATCHED: contract universalrouter is routerimmutables, iuniversalrouter, dispatcher, rewardscollector { modifier checkdeadline(uint256 deadline) { if (block.timestamp > deadline) revert transactiondeadlinepassed();
UNMATCHED: self.balanceof[msg.sender] -= _value self.balanceof[_to] += _value log transfer(msg.sender, _to, _value) return true   @external def transferfrom(_from : address, _to : address, _value : uint256) -> bool: """ @notice transfer `_value` tokens from `_from` to `_to` @param _from address the address which you want to send tokens from @param _to address the address which you want to transfer to @param _value uint256 the amount of tokens to be transferred @return bool success """ assert _to != zero_address   self.balanceof[_from] -= _value self.balanceof[_to] += _value self.allowances[_from][msg.sender] -= _value log transfer(_from, _to, _value) return true   @external def approve(_spender : address, _value : uint256) -> bool: """ @notice approve `_spender` to tra

## Embed invariants WITH errormessages

In [38]:
from sentence_transformers import SentenceTransformer
vectorizer = TfidfVectorizer()  
X = vectorizer.fit_transform(with_message).toarray()
code_BERT = embedding_model(with_message, "microsoft/codebert-base")
smart_BERT = embedding_model(with_message, "web3se/SmartBERT-v2")
model = SentenceTransformer("./smartbert-contrastive")
smart_finetuned = model.encode(with_message, convert_to_numpy=True, show_progress_bar=True)

Some weights of RobertaModel were not initialized from the model checkpoint at web3se/SmartBERT-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

In [25]:
import warnings

# Suppress FutureWarnings (like from sklearn deprecation)
warnings.filterwarnings("ignore", category=FutureWarning)

# Suppress UserWarnings (like from umap)
warnings.filterwarnings("ignore", category=UserWarning)


## Cluster invariants WITH errormessages

In [ ]:
cluster_run_all_with_metrics(X, "TF-IDF", with_message, with_msg = True, metrics = ["cosine"])
cluster_run_all_with_metrics(code_BERT, "codeBERT", with_message, with_msg = True, metrics = ["cosine"])
cluster_run_all_with_metrics(smart_BERT, "SmartBert", with_message, with_msg = True, metrics = ["cosine"])
cluster_run_all_with_metrics(smart_finetuned, "ReBERT", with_message, with_msg = True, metrics = ["cosine"])

## Embed invariants WITHOUT errormessages

In [28]:
from sentence_transformers import SentenceTransformer
vectorizer = TfidfVectorizer()  
X_without = vectorizer.fit_transform(without_message).toarray()
code_BERT_without = embedding_model(without_message, "microsoft/codebert-base")
smart_BERT_without = embedding_model(without_message, "web3se/SmartBERT-v2")
model = SentenceTransformer("./smartbert-contrastive")
smart_finetuned_without = model.encode(without_message, convert_to_numpy=True, show_progress_bar=True)

Some weights of RobertaModel were not initialized from the model checkpoint at web3se/SmartBERT-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Batches: 100%|██████████| 23/23 [00:15<00:00,  1.46it/s]


## Cluster invariants WITHOUT errormessages

In [ ]:
#code_BERT = embedding_model(with_message, "microsoft/codebert-base")
cluster_run_all_with_metrics(X_without, "TF-IDF", without_message, with_msg = False, metrics = ["cosine"])
cluster_run_all_with_metrics(code_BERT_without, "codeBERT", without_message, with_msg = False, metrics = ["cosine"])
cluster_run_all_with_metrics(smart_BERT_without, "SmartBert", without_message, with_msg = False, metrics = ["cosine"])
cluster_run_all_with_metrics(smart_finetuned_without, "ReBERT", without_message, with_msg = False, metrics = ["cosine"])